# NeurIPS 2026 Smart Buildings — Version 16: LQR / LQG

This is a **classical state-space feedback** experiment, independent from PID (V2/3), predictive search (V6/12), constrained QP resource allocation (V14), or black-box gain tuning (V15). It retains the V7/V15 thermodynamic actuator mapping and correct outdoor-air normalization, but **zone demand is determined by LQR rather than temperature-error heuristics**.

## State-space model (one zone, one 5-minute step)

`T[t+1] = a T[t] + (1-a) T_out[t] + b u[t] + d[t]`; `d[t+1] = d[t] + w[t]`, `y[t] = T[t] + v[t]`.

- `T`: zone temperature in Kelvin; `T_out`: outside temperature in Kelvin.
- `u`: abstract *signed* normalized thermal effort, positive for reheat, negative for cooling.
- `a = 0.995`, `b = 0.23 K / step` are **assumed**, not identified from the simulator; the real plant has nonlinearity, delays, coupling and saturation.
- The LQR gain solves the **scalar discrete algebraic Riccati equation** for `Q=1`, `R=4`, computed once on class initialization.
- LQG mode (`USE_KALMAN=True`) augments the state with unknown thermal drift. `ZoneKalman` is a two-state Kalman observer with a temperature measurement and bounded covariance/drift; it predicts using the previous *issued* effort, not a fictitious measured effective airflow.
- LQR-only ablation (`USE_KALMAN=False`) uses the raw denormalized zone temperature and zero drift.
- A zone is conditioned only near its real heating/cooling comfort band; there is **no energy-expensive midpoint tracking**. The regulator uses a boundary target, steady-state outdoor/drift compensation, and explicit effort bounds.

## Physical dispatch

`_zone_effort` gives one signed thermal request per zone. `_select_air_supply` chooses a cooling-only coil temperature and economizer mixing; `_control_zone_boxes` maps negative requests to VAV cooling flow and positive requests to reheat valve / hot-water demand. `_control_ahu` and `_control_hot_water` dispatch fan and boiler. All action indices and temperature normalizers come from building metadata.

The action contract remains `float32[-1,1]` continuous and `int32{0,1}` discrete, with a final official Kaggle inference-server cell. `reset()` clears the per-zone observer state between episodes. Runtime and memory scale linearly with the number of zones; no online Riccati or matrix inverse occurs.

## How to evaluate

Run `python Co_Kaggle/g8/submissions/test_version16.py` for synthetic interface, Riccati stability, Kalman convergence, LQR ablation, reset, weather normalization and performance checks.

For real evidence, compare **identical scenarios and 8,640-step horizons** against V7 (public score **0.03947**) and V15 (recorded score **0.04241**). Separately evaluate LQR-only vs LQG, sensor-noise drift robustness, outdoor climates, comfort-tail violations, energy/carbon, and full runtime. No official simulator or leaderboard score is claimed.

**Limitation:** Because the physical mapping from `u` to AHU/VAV commands is nonlinear and uncalibrated, this is a *locally LQR/LQG-derived hybrid controller*, not a globally optimal constrained LQG controller. It is intentionally a focused controller-family experiment, not an assumed improvement.


In [ ]:
"""Version 16: independent LQR / LQG zone regulation on physical HVAC dispatch.

A scalar discrete-time Riccati controller regulates each zone temperature;
optional two-state Kalman filters estimate temperature and unmodeled thermal
drift. Saturation, AHU switching, reheat and mixing remain physical mappings.
No online optimization or identified building model is claimed.
"""
import re
import numpy as np
from kaggle_evaluation import smart_buildings_inference_server


# Fixed, explicitly assumed thermal model (per 5-minute simulation step).
# These are plausible starting coefficients, NOT simulator-fitted parameters.
A_THERMAL = 0.995
B_THERMAL = 0.23  # K per five-minute signed normalized conditioning effort
PROCESS_TEMP_VARIANCE = 0.03**2
PROCESS_DRIFT_VARIANCE = 0.012**2
SENSOR_VARIANCE = 0.10**2

# Distinct physical-system and optimization parameters are deliberately kept
# separate. The actuator conversion constants live on TransferPolicy.

def scalar_discrete_lqr_gain(a=A_THERMAL, b=B_THERMAL, q=1.0, r=4.0):
    """Solve the stabilizing scalar discrete algebraic Riccati equation.

    For x[t+1] = a*x[t] + b*u[t], minimize sum(q*x^2 + r*u^2).
    No generic optimizer / SciPy solve occurs during inference.
    """
    if not all(np.isfinite(v) for v in (a, b, q, r)):
        raise ValueError("Nonfinite LQR coefficient")
    if not (0.0 <= a < 1.0 and b > 0.0 and q > 0.0 and r > 0.0):
        raise ValueError("Require 0<=a<1, b>0, q>0, r>0")
    c = r * (1.0 - a*a) - q*b*b
    p = (2.0*q*r) / (c + np.sqrt(c*c + 4.0*b*b*q*r))
    # The stable quadratic root is (sqrt(discriminant)-c)/(2*b^2).
    # The rationalized positive root avoids cancellation when c > 0.
    return float((a*b*p) / (r + b*b*p))


class ZoneKalman:
    """Two-state (temperature, unmodeled K/step drift) Kalman observer.

    State: [temperature_K, drift_K_per_step].
    Dynamics: temperature' = a*T + (1-a)*outside + b*u + drift.
    The process/measurement covariance is fixed; previous *issued*
    effort is used for prediction, not an assumed measured HVAC output.
    """

    def __init__(self, measured_kelvin):
        self.temperature = float(measured_kelvin)
        self.drift = 0.0
        self.p_tt = 0.04
        self.p_td = 0.0
        self.p_dd = 0.004
        self.previous_effort = 0.0
        self.previous_outside = float(measured_kelvin)

    def observe(self, measured_kelvin):
        """Predict through last action, assimilate sensor, bound uncertainty."""
        a, b = A_THERMAL, B_THERMAL
        predicted_t = (a*self.temperature
                       + (1.0-a)*self.previous_outside
                       + b*self.previous_effort + self.drift)
        predicted_d = self.drift
        ptt = (a*a*self.p_tt + 2.0*a*self.p_td + self.p_dd
               + PROCESS_TEMP_VARIANCE)
        ptd = a*self.p_td + self.p_dd
        pdd = self.p_dd + PROCESS_DRIFT_VARIANCE
        innovation = float(measured_kelvin) - predicted_t
        if abs(innovation) > 4.0:
            # Large jumps (setpoint changes, sensor restart) must not be
            # suppressed by the filter and cause unsafe delayed response.
            self.__init__(measured_kelvin)
            return self.temperature, self.drift
        innovation = float(np.clip(innovation, -2.0, 2.0))
        scale = ptt + SENSOR_VARIANCE
        kt, kd = ptt / scale, ptd / scale
        self.temperature = predicted_t + kt*innovation
        self.drift = float(np.clip(predicted_d + kd*innovation, -0.18, 0.18))
        # Kalman covariance update P = (I-KH)P_pred; H=[1,0].
        self.p_tt = float(np.clip((1.0-kt)*ptt, 1e-7, 1.0))
        self.p_dd = float(np.clip(pdd-kd*ptd, 1e-7, 0.1))
        self.p_td = float(np.clip((1.0-kt)*ptd,
                                  -0.99*np.sqrt(self.p_tt*self.p_dd),
                                   0.99*np.sqrt(self.p_tt*self.p_dd)))
        return self.temperature, self.drift


class TransferPolicy(smart_buildings_inference_server.Policy):
    """LQG zone regulator with safe physical AHU/VAV/hot-water dispatch.

    Action contract: continuous float32 in [-1, 1], discrete int32 in {0, 1}.
    All equipment topology and sensor locations come from episode metadata.
    """

    # Physical constants from the starter-kit action descriptions.
    MIN_SUPPLY_K = 285.0
    MAX_SUPPLY_K = 305.0
    MAX_RETURN_FRACTION = 0.70
    MIN_VAV_FRACTION = 0.10
    MIN_FAN_PRESSURE_PA = 200.0
    # A cooling-only AHU still admits at least 30% outside air.
    # Increasing intake beyond that floor also raises exhaust-fan power.
    MIN_ECONOMIZER_OUTDOOR_K = 287.0
    MIN_ECONOMIZER_RECIRC = 0.15
    # LQR and supervisory-layer constants. These are assumptions, not fitted
    # to the organizer's simulator; compare them to V7 on real rollouts.
    USE_KALMAN = True  # False gives the same controller without the observer.
    LQR_GAIN = scalar_discrete_lqr_gain()
    MIN_EFFORT = 0.08
    TRIGGER_MARGIN_K = 0.03
    TARGET_MARGIN_K = 0.25

    # Map signed, unitless control effort to the simulator's hybrid actuators.
    COOLING_OFFSET_K = 4.0
    COOLING_EFFORT_GAIN_K = 1.3
    FAN_PRESSURE_SPAN_PA = 1250.0
    WATER_WEATHER_GAIN = 0.35
    WATER_SHORTFALL_GAIN = 5.0

    def __init__(self, building_meta_data, action_spec):
        """Index observations and actions once, never during each time step."""
        super().__init__(building_meta_data, action_spec)
        meta = building_meta_data
        self.n_continuous = int(meta["n_continuous_action"])
        self.n_discrete = int(meta["n_discrete_action"])
        self.fields = {str(s): i for i, s in enumerate(meta["observation_field_names"])}
        self.supply_indices = list(map(int, meta["temp_indices"]))
        self.pressure_indices = list(map(int, meta["pressure_indices"]))
        self.recirc_indices = list(map(int, meta["recirculation_indices"]))
        self.water_indices = list(map(int, meta["water_indices"]))
        self.ahu_count = len(self.supply_indices)
        self.water_supply = self.water_indices[0] if self.water_indices else None
        self.water_pressure = (self.pressure_indices[self.ahu_count]
                               if len(self.pressure_indices) > self.ahu_count else None)
        self.water_run = self.n_discrete - 1

        self.comfort = tuple(map(float, meta["comfort_temp_window"]))
        self.eco = tuple(map(float, meta["eco_temp_window"]))
        self.morning = float(meta["morning_start_hour"])
        self.evening = float(meta["evening_start_hour"])
        normalizer = meta.get("observation_normalization_metadata", {}).get(
            "zone_air_temperature_sensor", {})
        self.zone_mean = float(normalizer.get("sample_mean", 0.0))
        self.zone_std = float(normalizer.get("sample_std", 1.0))
        if not np.isfinite(self.zone_std) or self.zone_std <= 0:
            self.zone_std = 1.0
        outdoor_stats = meta.get("observation_normalization_metadata", {}).get(
            "outside_air_temperature_sensor", {})
        self.outdoor_mean = float(outdoor_stats.get("sample_mean", float("nan")))
        self.outdoor_std = float(outdoor_stats.get("sample_std", float("nan")))

        def unit_order(key):
            match = re.search(r"\d+", str(key))
            return (int(match.group()) if match else -1, str(key))

        self.ahus = []
        assignments = meta["ahu_zone_assignments"]
        indices = meta["zone_action_indices"]
        for i, identifier in enumerate(sorted(assignments, key=unit_order)[:self.ahu_count]):
            zones = []
            for room in assignments[identifier]:
                vav = str(room).replace("room_", "VAV_")
                temp = self.fields.get(vav + "_zone_air_temperature_sensor")
                if temp is None:
                    continue
                actuators = indices.get(vav, {})
                zones.append({
                    "temperature": temp,
                    "low": self.fields.get(vav + "_zone_air_heating_temperature_setpoint_sensor"),
                    "high": self.fields.get(vav + "_zone_air_cooling_temperature_setpoint_sensor"),
                    "damper": actuators.get("supply_air_damper_percentage_command"),
                    "valve": actuators.get("reheat_valve_percentage_command"),
                })
            self.ahus.append({
                "zones": zones,
                "run": i,
                "supply": self.supply_indices[i],
                "pressure": (self.pressure_indices[i]
                             if i < len(self.pressure_indices) else None),
                "recirc": (self.recirc_indices[i]
                           if i < len(self.recirc_indices) else None),
                "outdoor": self.fields.get(
                    f"ahs_{identifier}_outside_air_temperature_sensor"),
            })
        self.reset()

    def reset(self):
        """Clear all zone observers and previous actions at episode boundary."""
        self.zone_filters = {}

    @staticmethod
    def _clamp(value, low, high):
        return float(np.clip(value, low, high))

    @classmethod
    def _fraction_action(cls, fraction):
        """Convert physical 0..1 actuator opening to normalized action."""
        return 2.0 * cls._clamp(fraction, 0.0, 1.0) - 1.0

    @staticmethod
    def _temperature_action(kelvin):
        """Supply air 285..305 K maps to -1..1."""
        return float(np.clip((kelvin - 295.0) / 10.0, -1.0, 1.0))

    def _value(self, obs, index):
        """Read an optional sensor without indexing outside the observation."""
        if index is None or not 0 <= index < len(obs):
            return float("nan")
        return float(obs[index])

    def _outside_temperature(self, obs, index):
        """Denormalize outdoor air; never treat a z-score as Kelvin."""
        raw = self._value(obs, index)
        if not np.isfinite(raw):
            return float("nan")
        if (np.isfinite(self.outdoor_mean) and
                np.isfinite(self.outdoor_std) and self.outdoor_std > 0.0):
            kelvin = raw * self.outdoor_std + self.outdoor_mean
        else:
            kelvin = raw
        return kelvin if 210.0 <= kelvin <= 330.0 else float("nan")

    def _schedule_fallback(self, obs):
        """Fall back to the occupied/eco schedule if zone setpoints are absent."""
        def clock(sin_name, cos_name, period):
            sine = self._value(obs, self.fields.get(sin_name))
            cosine = self._value(obs, self.fields.get(cos_name))
            if not (np.isfinite(sine) and np.isfinite(cosine)):
                return None
            return float(np.arctan2(sine, cosine) % (2 * np.pi)) * period / (2 * np.pi)

        hour = clock("hod_sin_000", "hod_cos_000", 24.0)
        day = clock("dow_sin_000", "dow_cos_000", 7.0)
        weekday = day is not None and int(day) < 5
        occupied = (weekday and hour is not None
                    and self.morning <= hour < self.evening)
        return self.comfort if occupied else self.eco

    def _read_zone(self, obs, zone, fallback):
        """Decode one zone (Kelvin); return None for an invalid temperature.

        The temperature observation is standardized. Setpoint observations
        are already in Kelvin and take precedence over schedule fallback.
        """
        raw = self._value(obs, zone["temperature"])
        temperature = raw * self.zone_std + self.zone_mean
        if not np.isfinite(temperature) or not 240.0 <= temperature <= 340.0:
            return None
        low = self._value(obs, zone["low"])
        high = self._value(obs, zone["high"])
        if not (np.isfinite(low) and np.isfinite(high)
                and 245.0 <= low < high <= 335.0 and high - low <= 35.0):
            low, high = fallback
        return {
            "zone": zone,
            "temp": temperature,
            "hot": max(0.0, temperature - high),
            "cold": max(0.0, low - temperature),
            "low": low,
            "high": high,
        }

    def _make_idle_actions(self):
        """A zero action is half-open, so explicitly set idle commands."""
        continuous = np.full(self.n_continuous, -1.0, dtype=np.float32)
        discrete = np.zeros(self.n_discrete, dtype=np.int32)
        for index in self.supply_indices:
            continuous[index] = 1.0  # Warmest: cooling coil effectively idle.
        for index in self.recirc_indices:
            continuous[index] = 1.0  # Max return air, physically capped at 70%.
        return continuous, discrete

    def _estimate_zone_state(self, zone, measured_kelvin):
        """Return estimated temperature, thermal drift and optional observer.

        Without the Kalman filter, this is ordinary measured-state LQR.
        """
        if not self.USE_KALMAN:
            return measured_kelvin, 0.0, None

        key = zone["temperature"]
        observer = self.zone_filters.get(key)
        if observer is None:
            observer = ZoneKalman(measured_kelvin)
            self.zone_filters[key] = observer
            return measured_kelvin, 0.0, observer

        estimated_temp, drift = observer.observe(measured_kelvin)
        return estimated_temp, drift, observer

    def _comfort_target(self, low, high, estimated, predicted):
        """Select an *edge* of the acceptable band, or request no control.

        A deadband preserves energy; LQR tracking the band's midpoint at all
        times would waste energy without improving the competition's reward.
        """
        if max(estimated, predicted) > high - self.TRIGGER_MARGIN_K:
            return high - self.TARGET_MARGIN_K, -1  # Cooling.
        if min(estimated, predicted) < low + self.TRIGGER_MARGIN_K:
            return low + self.TARGET_MARGIN_K, 1  # Reheat.
        return None, 0

    def _zone_effort(self, status, outdoor_kelvin):
        """Compute one signed effort: negative=cooling, positive=reheat.

        Feedback is genuine scalar discrete-time LQR. Add a steady-state
        disturbance offset and apply physical deadband/saturation *after*
        computing the unconstrained requested control.
        """
        zone = status["zone"]
        measured = status["temp"]
        estimate, drift, observer = self._estimate_zone_state(zone, measured)

        # When the outside sensor is invalid, assume ambient equals zone
        # temperature. This disables uncertain weather feed-forward.
        ambient = outdoor_kelvin if np.isfinite(outdoor_kelvin) else measured
        predicted = (A_THERMAL * estimate
                     + (1.0 - A_THERMAL) * ambient + drift)
        target, direction = self._comfort_target(
            status["low"], status["high"], estimate, predicted)

        # An unavailable damper/valve cannot implement the requested input.
        # Do not turn on an AHU merely to chase an impossible thermal action.
        can_condition = zone["damper"] is not None
        can_reheat = (zone["valve"] is not None
                      and 0 <= self.water_run < self.n_discrete)
        if target is None or not can_condition or (direction > 0 and not can_reheat):
            effort = 0.0
        else:
            steady_state_effort = (
                (1.0 - A_THERMAL) * (target - ambient) - drift
            ) / B_THERMAL
            lqr_feedback = -self.LQR_GAIN * (estimate - target)
            requested_effort = lqr_feedback + steady_state_effort

            if direction < 0:
                effort = self._clamp(requested_effort, -1.0, 0.0)
            else:
                effort = self._clamp(requested_effort, 0.0, 1.0)
            if abs(effort) <= self.MIN_EFFORT:
                effort = 0.0

        if observer is not None:
            # The observer predicts with issued *abstract* control effort;
            # it cannot assume the mapped HVAC hardware delivers it exactly.
            observer.previous_effort = effort
            observer.previous_outside = ambient
        return effort

    def _select_air_supply(self, zones, outside_k):
        """Cooling-only AHU with free-cooling mixing; heating uses VAV reheat."""
        strongest_cooling = max(0.0, max(-z["effort"] for z in zones))
        return_k = float(np.mean([z["temp"] for z in zones]))
        if strongest_cooling < self.MIN_EFFORT:
            coil_setpoint_k = self.MAX_SUPPLY_K
        else:
            cooling_rooms = [z["temp"] for z in zones
                             if z["effort"] < -self.MIN_EFFORT]
            mean_hot_k = float(np.mean(cooling_rooms))
            coil_setpoint_k = self._clamp(
                mean_hot_k - self.COOLING_OFFSET_K
                - self.COOLING_EFFORT_GAIN_K * strongest_cooling,
                self.MIN_SUPPLY_K, 297.0)
        if not np.isfinite(outside_k):
            outside_k = return_k
        return_fraction = self.MAX_RETURN_FRACTION
        if (strongest_cooling >= self.MIN_EFFORT
                and self.MIN_ECONOMIZER_OUTDOOR_K <= outside_k < return_k-1.5):
            min_mix = (self.MIN_ECONOMIZER_RECIRC*return_k
                       + (1.0-self.MIN_ECONOMIZER_RECIRC)*outside_k)
            if min_mix <= coil_setpoint_k:
                required = (coil_setpoint_k-outside_k)/(return_k-outside_k)
                return_fraction = self._clamp(
                    required, self.MIN_ECONOMIZER_RECIRC, self.MAX_RETURN_FRACTION)
        mixed_k = return_fraction*return_k+(1.0-return_fraction)*outside_k
        return coil_setpoint_k, return_fraction, min(mixed_k, coil_setpoint_k)

    def _control_zone_boxes(self, statuses, delivered, hot_water_demands, continuous):
        """Translate signed effort to damper and reheat valve commands.

        This nonlinear conversion is an actuator approximation; b in the
        Riccati model has not been calibrated to realized heat transfer.
        """
        total_damper = 0.0
        peak_damper = 0.0

        for status in statuses:
            zone = status["zone"]
            effort = status["effort"]
            temperature = status["temp"]

            if effort < -self.MIN_EFFORT:
                cooling_headroom = max(2.5, temperature - delivered)
                damper = self._clamp(
                    max(0.12 + 3.2 * (-effort) / cooling_headroom,
                        0.12 + 0.10 * status["hot"]),
                    self.MIN_VAV_FRACTION, 0.98)
            elif effort > self.MIN_EFFORT:
                damper = self._clamp(0.20 + 0.52 * effort, 0.18, 0.88)
                valve = self._clamp(0.12 + 0.82 * effort, 0.0, 1.0)
                # _zone_effort already verified that reheat is available.
                continuous[zone["valve"]] = self._fraction_action(valve)
                hot_water_demands.append(
                    (max(status["cold"], 0.25 * effort), valve))
            else:
                damper = self.MIN_VAV_FRACTION

            if zone["damper"] is not None:
                continuous[zone["damper"]] = self._fraction_action(damper)
            total_damper += damper
            peak_damper = max(peak_damper, damper)

        return total_damper / len(statuses), peak_damper

    def _control_ahu(self, ahu, statuses, outdoor, continuous, discrete, water):
        """Run only if a zone needs conditioning; never call the AHU to heat."""
        if not statuses:
            return
        if not any(abs(z["effort"]) > self.MIN_EFFORT for z in statuses):
            return
        target, recirc, delivered = self._select_air_supply(statuses, outdoor)
        mean_damper, peak_damper = self._control_zone_boxes(
            statuses, delivered, water, continuous)
        discrete[ahu["run"]] = 1
        continuous[ahu["supply"]] = self._temperature_action(target)
        if ahu["recirc"] is not None:
            continuous[ahu["recirc"]] = self._fraction_action(recirc)
        if ahu["pressure"] is not None:
            # Use both typical load and worst VAV. A badly overheated room
            # must not disappear in the average of many neutral zones.
            # The fan model uses airflow ~ sqrt(static pressure).
            demand_proxy = max(mean_damper, 0.50 * peak_damper)
            load = self._clamp(
                (demand_proxy - self.MIN_VAV_FRACTION) / 0.65, 0.0, 1.0)
            pressure_pa = (self.MIN_FAN_PRESSURE_PA
                           + self.FAN_PRESSURE_SPAN_PA * np.sqrt(load))
            continuous[ahu["pressure"]] = self._clamp(
                pressure_pa / 1000.0 - 1.0, -1.0, 1.0)

    def _control_hot_water(self, water, outdoor_values, continuous, discrete):
        """Weather-compensated hot-water reset, independent of cooling mode.

        Cold outdoor air forces a lower mixed-air temperature because the
        simulator caps recirculation at 70%; compensate with hotter reheat
        water. Only start the pump if a VAV actually requests heating.
        """
        if not water:
            return
        if 0 <= self.water_run < self.n_discrete:
            discrete[self.water_run] = 1
        else:
            return
        worst = max(shortfall for shortfall, _ in water)
        average_valve = sum(valve for _, valve in water) / len(water)
        outside = (sum(outdoor_values) / len(outdoor_values)
                   if outdoor_values else 294.0)
        weather_compensation = self.WATER_WEATHER_GAIN * max(0.0, 294.0 - outside)
        demand_compensation = self.WATER_SHORTFALL_GAIN * worst
        water_kelvin = self._clamp(
            318.0 + weather_compensation + demand_compensation, 320.0, 349.0)
        if self.water_supply is not None:
            continuous[self.water_supply] = self._fraction_action(
                (water_kelvin - 310.0) / 40.0)
        if self.water_pressure is not None:
            continuous[self.water_pressure] = self._fraction_action(
                self._clamp(0.38 + 0.52 * average_valve, 0.38, 0.95))

    def get_action(self, obs):
        """Update observer -> compute LQR effort -> dispatch HVAC commands.

        O(number of zones + AHUs) time and O(number of zones) state.
        """
        obs = np.asarray(obs, dtype=np.float64).reshape(-1)
        continuous, discrete = self._make_idle_actions()
        fallback = self._schedule_fallback(obs)
        hot_water_demands, outdoors = [], []
        for ahu in self.ahus:
            statuses = []
            for zone in ahu["zones"]:
                status = self._read_zone(obs, zone, fallback)
                if status is None:
                    # A missed measurement invalidates the observer's time
                    # step and last-action history. Reinitialize on recovery.
                    self.zone_filters.pop(zone["temperature"], None)
                    continue
                statuses.append(status)
            outdoor = self._outside_temperature(obs, ahu["outdoor"])
            if np.isfinite(outdoor) and 210.0 <= outdoor <= 330.0:
                outdoors.append(outdoor)
            else:
                outdoor = float("nan")
            for status in statuses:
                status["effort"] = self._zone_effort(status, outdoor)
            self._control_ahu(
                ahu, statuses, outdoor, continuous, discrete, hot_water_demands)
        self._control_hot_water(hot_water_demands, outdoors, continuous, discrete)
        np.nan_to_num(continuous, copy=False, nan=-1.0, posinf=1.0, neginf=-1.0)
        np.clip(continuous, -1.0, 1.0, out=continuous)
        return {"continuous_action": continuous, "discrete_action": discrete}


In [ ]:
# Official Kaggle evaluator; do not substitute an offline CSV writer.
smart_buildings_inference_server.SmartBuildingsInferenceServer(TransferPolicy).run()
